## 欢迎回到 Python Notebooks！

想我了吗？？

### 欢迎来到第四周第二天——LangGraph 入门！

In [1]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random


In [ ]:
# 一些有用的常量

nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [ ]:
# 我们最熟悉的第一步！顺便说一下，Crew 之前一直在帮我们做这个。
load_dotenv(override=True)

In [ ]:
def shout(text: Annotated[str, "something to be shouted"]) -> str:
    print(text.upper())
    return text.upper()

shout("hello")

### 关于 "Annotated" 的一点说明

你可能已经知道：类型提示是 Python 的一个特性，可以指定某个变量的类型：

`my_favorite_things: List`

但你可能不知道：

你还可以使用一种叫做 "Annotated" 的方式来添加额外的信息，供其他人使用：

`my_favorite_things: Annotated[List, "these are a few of mine"]`

LangGraph 要求我们在定义 State 对象时使用这个特性。

它需要我们告诉它：**应该调用什么函数来用新值更新 State**。

这个函数叫做 **reducer（合并器）**。

LangGraph 提供了一个默认的 reducer 叫做 `add_messages`，它可以处理最常见的情况。

这应该能解释为什么 State 看起来是这个样子。

### 第 1 步：定义 State 对象

你可以使用任何 Python 对象；但最常见的是使用 TypedDict 或 Pydantic BaseModel。

In [5]:

class State(BaseModel):
        
    messages: Annotated[list, add_messages]


### 第 2 步：用这个 State 类启动 Graph Builder

In [6]:
graph_builder = StateGraph(State)

### 第 3 步：创建一个 Node

一个 node 可以是任意 Python 函数。

我们之前设置的 reducer 会被自动调用，用来将当前响应与之前的响应合并。

In [ ]:
def our_first_node(old_state: State) -> State:

    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{"role": "assistant", "content": reply}]

    new_state = State(messages=messages)

    return new_state

graph_builder.add_node("first_node", our_first_node)

### 第 4 步：创建 Edges（边）

In [ ]:
graph_builder.add_edge(START, "first_node")
graph_builder.add_edge("first_node", END)

### 第 5 步：编译 Graph

In [9]:
graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### 就这样！开始展示！

In [ ]:
def chat(user_input: str, history):
    message = {"role": "user", "content": user_input}
    messages = [message]
    state = State(messages=messages)
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

### 为什么我要先展示这个？

为了说明一个重点：**LangGraph 的核心是 Python 函数——它不一定需要涉及 LLM！！**

现在我们再来一遍这 5 个步骤，但一气呵成：

In [ ]:
# 第 1 步：定义 State 对象
class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
# 第 2 步：用这个 State 类启动 Graph Builder
graph_builder = StateGraph(State)

In [ ]:
# 第 3 步：创建一个 Node

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

In [ ]:
# 第 4 步：创建 Edges（边）
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [ ]:
# 第 5 步：编译 Graph
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### 搞定！接下来，让我们做这个：

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content


gr.ChatInterface(chat, type="messages").launch()